# Day 42 — Unsupervised learning: KMeans & anomaly detection
Objectives:
- Cluster data with KMeans and interpret clusters.
- Evaluate with inertia/silhouette (caveats).
- Simple anomaly detection (IsolationForest/LocalOutlierFactor).


In [ ]:
from sklearn.datasets import make_blobs
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
X, y_true = make_blobs(n_samples=600, centers=3, cluster_std=1.2, random_state=42)
km = KMeans(n_clusters=3, n_init='auto', random_state=42).fit(X)
sil = silhouette_score(X, km.labels_)
sil
plt.scatter(X[:,0], X[:,1], c=km.labels_, cmap='viridis', s=15)
plt.title('KMeans clusters'); plt.show()


In [ ]:
# Anomaly detection
from sklearn.ensemble import IsolationForest
iso = IsolationForest(contamination=0.02, random_state=42).fit(X)
scores = iso.decision_function(X)
pred = iso.predict(X)  # -1 outlier, 1 inlier
(pred==-1).mean()


## How to use this notebook

Select the `Python (ds60sqlpy)` kernel, start at the first cell, and
write each prediction before execution. Keep attempts in the
provided scratch cell or new cells. Restart the kernel and run from
the top before calling the work reproducible.

## Concept lab — unsupervised geometry, cluster stability, and anomaly ranking

### Mental model

Unsupervised algorithms optimize a mathematical objective without known
target labels. K-Means alternates between assigning points to the nearest
centroid and recomputing centroid means. It therefore favors roughly
spherical, similarly scaled clusters under Euclidean distance.

Anomaly detectors produce a score or ranking relative to the fitted
reference distribution. A contamination setting often converts that
ranking into a fixed fraction of labels; it does not prove those cases
are errors or threats. Stability, synthetic controls, and domain review
are essential evidence.

### Read the API before running it

- **`KMeans(n_clusters=k, n_init=..., random_state=...)`:** declares cluster count and repeated initialization so a local optimum is not mistaken for truth.
- **`silhouette_score(X, labels)`:** compares cohesion and separation under the selected distance; it needs at least two nontrivial clusters.
- **`decision_function(X)`:** returns a continuous anomaly score; inspect ordering before applying a threshold.

For every call, identify input data, learned state, returned value,
and a check that can fail. That habit prevents a successful cell
from being mistaken for a correct analysis.

### Focused example A — show that feature scale changes K-Means geometry

**Predict first:** write down the expected shape, type, ordering, or
direction of the result. Then run the next cell.

**Assumption:** Standard-deviation scaling represents the intended similarity, rather than erasing meaningful units.

In [ ]:
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(4201)
left = np.column_stack([rng.normal(-2, 0.4, 80), rng.normal(0, 100, 80)])
right = np.column_stack([rng.normal(2, 0.4, 80), rng.normal(0, 100, 80)])
X = np.vstack([left, right])

raw_labels = KMeans(n_clusters=2, n_init=10, random_state=4201).fit_predict(X)
scaled_labels = KMeans(n_clusters=2, n_init=10, random_state=4201).fit_predict(
    StandardScaler().fit_transform(X)
)
print({"raw_agreement_with_x_sign": np.mean(raw_labels == (X[:, 0] > 0)),
       "scaled_unique_labels": np.unique(scaled_labels).size})

**Expected observation:** The large-unit noise dimension can dominate raw Euclidean distance; scaling changes assignments.

Do not force exact equality for estimates based on samples. Record
the seed, sample size, tolerance, and metric where they matter.

### Focused example B — treat anomaly output as a ranking, not a verdict

This example changes one important condition. Predict how and why
the result should differ from Example A.

**Assumption:** The synthetic extremes are only a control; domain review determines whether a real unusual case is harmful or valuable.

In [ ]:
import numpy as np
from sklearn.ensemble import IsolationForest

rng = np.random.default_rng(4202)
ordinary = rng.normal(size=(200, 2))
obvious = np.array([[8.0, 8.0], [-8.0, -8.0]])
X = np.vstack([ordinary, obvious])
detector = IsolationForest(contamination=0.02, random_state=4202).fit(X)
scores = detector.decision_function(X)
most_unusual = np.argsort(scores)[:5]
print({"lowest_score_indices": most_unusual.tolist(),
       "injected_indices": [200, 201]})
assert {200, 201}.issubset(set(most_unusual))

**Expected observation:** The injected extremes rank among the lowest scores, but the threshold also labels a configured fraction of ordinary data.

### Debugging and practice ramp

**Common mistake:** Naming clusters as real customer types or anomalies as fraud solely because the algorithm produced labels.

**Diagnostic:** Check scaling, seed/init stability, cluster sizes, centroid movement, score distribution, synthetic controls, and representative reviewed cases.

| Stage | Action | Evidence |
|---|---|---|
| Recall | Define unsupervised geometry, cluster stability, and anomaly ranking in your own words and identify its input and output. | A definition that does not rely on the library name. |
| Predict | Predict the examples before execution, including shape and direction. | A written prediction and an explanation of any mismatch. |
| Implement | Recreate one example with a changed but valid input. | Code plus an assertion for the central invariant. |
| Debug | Trigger the named mistake or edge case intentionally. | The observed symptom and the smallest diagnostic that isolates it. |
| Transfer | Apply the idea to a different local dataset or decision. | A stated assumption, metric, and reason the method is suitable. |

**Stop condition:** Do not operationalize unsupervised labels without a semantic validation and false-positive handling plan.

Continue to the numbered practice only after you can explain both
examples without rereading their code.

## Learner exercises and progressive hints

1. Try several values of `k` and plot inertia versus `k` (the elbow plot).

**Verify:** Practice 1 — unsupervised geometry, cluster stability, and anomaly ranking — for each declared k, print inertia, cluster sizes, and seed and save a labeled elbow curve; assert inertia is non-increasing and do not call the visual bend proof of true classes.

2. Compute silhouette scores for `k` from 2 through 6 and discuss the result.

**Verify:** Practice 2 — unsupervised geometry, cluster stability, and anomaly ranking — for k=2..6 on identical scaled rows, print silhouette score and cluster sizes, choose the largest score only after checking no tiny/empty cluster, and repeat with at least three seeds to report stability.

3. Compare Isolation Forest with Local Outlier Factor.

**Verify:** Practice 3 — unsupervised geometry, cluster stability, and anomaly ranking — on identical scaled data and declared contamination/neighbors, print each method's continuous anomaly score and top-n row IDs; report overlap and inspect injected normal/outlier controls instead of equating labels blindly.

### Progressive hints

1. Include `k=1` for inertia, but not for silhouette. Record the same fitted
   model's `inertia_` rather than fitting again accidentally.
2. The maximum is a candidate, not an unquestionable answer. Compare the plot
   and whether known generated structure is recovered.
3. Both label outliers as `-1`, but LOF usually uses `fit_predict` for the
   training set. Match their `contamination` values before comparing counts.

### Additional mastery practice

Make unsupervised assumptions observable through scaling, stability, domain checks, and synthetic controls. A cluster ID or anomaly label is not ground truth.

Predict or plan before you run code. Use the hint only after an honest
attempt, and record the evidence that would prove your result correct.

4. **Scaling sensitivity:** Create two features with equal structure but scales of 1 and 10,000. Compare K-Means assignments before and after standardization.
   **Progressive hint:** Euclidean distance squares numeric differences, so the large-unit feature dominates unless that weighting is intentional.

**Verify:** Scaling sensitivity — print feature ranges and K-Means assignments before/after StandardScaler, then report adjusted Rand agreement; verify the 10,000-scale feature dominates the unscaled distance calculation.

5. **Cluster stability:** Refit K-Means across at least ten seeds and bootstrap samples. Compare inertia, silhouette, and assignment agreement without assuming numeric cluster labels line up.
   **Progressive hint:** Labels can permute. Use adjusted Rand index or align centers before comparing assignments.

**Verify:** Cluster stability — over at least ten seeds and bootstrap samples, print inertia, silhouette, cluster sizes, and label-aligned adjusted Rand scores; report distributions rather than comparing raw numeric labels.

6. **Anomaly validation without labels:** Design an evaluation plan for an anomaly detector when historical anomaly labels are incomplete. Include synthetic injection, review capacity, and contamination sensitivity.
   **Progressive hint:** Use multiple evidence sources: known incidents, injected anomalies, top-k expert review, and stability across reasonable settings.

**Verify:** Anomaly validation without labels — deliver an evaluation table naming synthetic anomaly fixtures, domain-review sample/precision-at-k, review capacity, contamination sweep, stability metric, owner, and a stop threshold for each check.

Before opening the reference solution, explain the relevant assumption,
failure mode, and validation check for every answer.

In [ ]:
# Expanded mastery lab scratch space
#
# Keep the official solution closed until you have attempted each task.
# Add small assertions, shape checks, or metric comparisons as evidence.

# Practice 4 — Scaling sensitivity


# Practice 5 — Cluster stability


# Practice 6 — Anomaly validation without labels
